In [1]:
!pip install supabase pandas python-dotenv pytz

Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached hpack-4.1.0-py3-none-any.whl (34 kB)
Using cached hyperframe-6.1.0-py

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To 

In [3]:
import os
from datetime import datetime
import pytz
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

In [4]:
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL,SUPABASE_KEY)

In [5]:
def current_ist_date():
    IST = pytz.timezone("Asia/Kolkata")
    return datetime.now(IST).strftime("%Y-%m-%d")

In [6]:
print("Current IST Date:",current_ist_date())

Current IST Date: 2026-04-11


In [7]:
def create_classroom(class_name, code="1234", daily_limit=10):
    existing = (
        supabase.table("classroom_settings")
        .select("*")
        .eq("class_name",class_name)
        .execute()
        .data
    )
    
    if existing:
        print(f"Class {class_name} already exists")
        return
    
    supabase.table("classroom_settings").insert({
        "class_name":class_name,
        "code":code,
        "daily_limit":daily_limit,
        "is_open":True
    }).execute()
    
    print(f"Classroom {class_name} created successfully")
    

In [9]:
create_classroom("Democlass1",code="4321",daily_limit=5)

Class Democlass1 already exists


In [ ]:
def add_student(class_name, roll_number, name):
    existing = (
        supabase.table("roll_map")
        .select("*")
        .eq("class_name",class_name)
        .eq("roll_number",roll_number)
        .execute()
        .data
    )
    
    if existing:
        print(f"Roll number {roll_number} already assigned to {existing[0]['name']}")
        return
    
    supabase.table("roll_map").insert({
        "class_name":class_name,
        "roll_number":roll_number,
        "name":name
    }).execute()
    
    print(f"Added {name} Roll {roll_number} to class {class_name}")
    

In [ ]:
add_student("Democlass1","1","Ali")
add_student("Democlass1","2","Ghazu")
add_student("Democlass1","3","aw")

Added Ali Roll 1 to class Democlass1
Added Ghazu Roll 2 to class Democlass1
Added aw Roll 3 to class Democlass1


In [ ]:
def mark_attendance(class_name, roll_number, code):
    
    today = current_ist_date()
    settings = (
        supabase.table("classroom_settings")
        .select("*")
        .eq("class_name",class_name)
        .execute()
        .data[0]
    )
    
    if code != settings["code"]:
        print("Incorrect code entered")
        return
    
    
    existing = (
        supabase.table("attendance")
        .select("*")
        .eq("class_name",class_name)
        .eq("roll_number",roll_number)
        .eq("date",today)
        .execute()
        .data
    )
    
    if existing:
        print("Attendance already marked today")
        return
    
    count_today = (
        supabase.table("attendance")
        .select("*",count="exact")
        .eq("class_name",class_name)
        .eq("date",today)
        .execute()
        .count
    )    
    
    if count_today>settings["daily_limit"]:
        print("Daily attendance limit reached")
        return
    
    student = (
        supabase.table("roll_map")
        .select("*")
        .eq("class_name",class_name)
        .eq("roll_number",roll_number)
        .execute()
        .data
    )
    
    if not student:
        print("Roll no not registered")
        return
    
    name = student[0]["name"]
    
    supabase.table("attendance").insert({
        "class_name":class_name,
        "roll_number":roll_number,
        "name":name,
        "date":today
    }).execute()
    
    print(f"Attendance recorded for {name} ({roll_number})")

In [ ]:
mark_attendance("Democlass1","1","4321")

Attendance already marked today


In [ ]:
mark_attendance("Democlass1","5","4321")

Roll no not registered


In [ ]:
def attendance_matrix(class_name):
    records = (
        supabase.table("attendance")
        .select("*")
        .eq("class_name",class_name)
        .order("date",desc=True)
        .execute()
        .data
    )
    
    if not records:
        print("No attendance found. ")
        return
    
    df = pd.DataFrame(records)
    df["status"] = "P"
    
    pivot_df = df.pivot_table(
        index = ["roll_number", "name"],
        columns = "date",
        values = "status",
        aggfunc="first",
        fill_value="A"
    ).reset_index()
    
    return pivot_df    

In [10]:
attendance_matrix("Democlass1")

date,roll_number,name,2026-04-06
0,1,Ali,P


In [11]:
def attendance_analytics(class_name):
    records = supabase.table("attendance").select("*").eq("class_name", class_name).execute().data
    if not records:
        print("No attendance found")
        return

    df = pd.DataFrame(records)
    df["status"] = "P"
    pivot_df = df.pivot_table(index=["roll_number","name"], columns="date", values="status", aggfunc="first", fill_value="A").reset_index()
    
    date_cols = pivot_df.columns[2:]
    pivot_df["Present_Count"] = pivot_df[date_cols].apply(lambda row: sum(val=="P" for val in row), axis=1)
    pivot_df["Attendance %"] = (pivot_df["Present_Count"]/len(date_cols)*100).round(2)
    
    print("Top 3 students:")
    print(pivot_df.sort_values("Attendance %", ascending=False).head(3))
    
    print("\nBottom 3 students:")
    print(pivot_df.sort_values("Attendance %").head(3))


In [12]:
attendance_analytics("Democlass1")

Top 3 students:
date  roll_number name 2026-04-06  Present_Count  Attendance %
0               1  Ali          P              1         100.0

Bottom 3 students:
date  roll_number name 2026-04-06  Present_Count  Attendance %
0               1  Ali          P              1         100.0
